In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from functools import reduce

print("1. Carregando dados do Data Lake...")
arquivos = dbutils.fs.ls("/Volumes/workspace/default/parquet/")
lista_dfs = [spark.read.parquet(f.path) for f in arquivos if f.name.endswith(".parquet")]
df_spark = reduce(lambda df1, df2: df1.unionByName(df2, allowMissingColumns=True), lista_dfs)

print("2. Mapeamento SPADL e Correção de Minutos...")
# Ações e posições
df_acoes = df_spark.withColumn('is_desarme', F.when((F.col('type_id') == 9) & (F.col('result_id') == 1), 1).otherwise(0)) \
                   .withColumn('is_driblado', F.when((F.col('type_id') == 9) & (F.col('result_id') == 0), 1).otherwise(0)) \
                   .withColumn('is_interceptacao', F.when((F.col('type_id') == 10) & (F.col('result_id') == 1), 1).otherwise(0)) \
                   .withColumn('is_corte', F.when(F.col('type_id') == 18, 1).otherwise(0)) \
                   .withColumn('is_falta', F.when(F.col('type_id') == 8, 1).otherwise(0)) \
                   .withColumn('is_keeper', F.when(F.col('type_id').between(14, 17), 1).otherwise(0)) \
                   .withColumn('is_chute', F.when(F.col('type_id') == 11, 1).otherwise(0))

# Granularidade por Jogo
df_scout_jogo = df_acoes.groupBy('game_id', 'player', 'team_id').agg(
    F.sum('is_desarme').alias('desarmes'), F.sum('is_interceptacao').alias('interceptacoes'),
    F.sum('is_corte').alias('cortes'), F.sum('is_falta').alias('faltas'),
    F.sum('is_driblado').alias('driblado'), F.sum('is_keeper').alias('acoes_goleiro'),
    F.sum('is_chute').alias('finalizacoes')
)

# Correção de Minutos e Posse de Bola
janela_jogo = Window.partitionBy('game_id')
df_posse = df_spark.filter(F.col('type_id') == 0).groupBy('game_id', 'team_id').count().withColumnRenamed('count', 'passes_time') \
                   .withColumn('passes_totais', F.sum('passes_time').over(janela_jogo)) \
                   .withColumn('posse_bola', (F.col('passes_time') / F.col('passes_totais')) * 100)

df_minutos_jogo = df_spark.groupBy('game_id', 'player', 'team_id', 'period_id') \
                          .agg((F.max('time_seconds') / 60).alias('min')) \
                          .groupBy('game_id', 'player', 'team_id').agg(F.sum('min').alias('minutos_jogados'))

df_game_log = df_minutos_jogo.join(df_scout_jogo, on=['game_id', 'player', 'team_id'], how='inner') \
                              .join(df_posse.select('game_id', 'team_id', 'posse_bola'), on=['game_id', 'team_id'], how='left')

# Injeção da Temporada (Extraída dos metadados)
df_caminhos = spark.read.parquet("/Volumes/workspace/default/parquet/") \
    .select('game_id', F.col('_metadata.file_path').alias('file_path')) \
    .withColumn('temporada', F.regexp_extract(F.col('file_path'), r'_(\d{4})', 1)) \
    .select('game_id', 'temporada').distinct()
df_game_log_temp = df_game_log.join(df_caminhos, on='game_id', how='inner')

print("3. Construindo Features de Contexto para ML...")
janela_estado_jogo = Window.partitionBy('game_id', 'period_id').orderBy('time_seconds')
df_ml_contexto = df_spark.withColumn('acao_anterior_id', F.lag('type_id', 1).over(janela_estado_jogo)) \
                         .withColumn('x_origem_anterior', F.lag('start_x', 1).over(janela_estado_jogo)) \
                         .withColumn('y_origem_anterior', F.lag('start_y', 1).over(janela_estado_jogo)) \
                         .withColumn('x_destino_anterior', F.lag('end_x', 1).over(janela_estado_jogo)) \
                         .withColumn('y_destino_anterior', F.lag('end_y', 1).over(janela_estado_jogo)) \
                         .withColumn('time_anterior_id', F.lag('team_id', 1).over(janela_estado_jogo)) \
                         .withColumn('delta_tempo_segundos', F.col('time_seconds') - F.lag('time_seconds', 1).over(janela_estado_jogo)) \
                         .withColumn('recuperou_posse', F.when(F.col('team_id') != F.col('time_anterior_id'), 1).otherwise(0))

display(
    df_game_log_temp.filter(F.col('player').contains('Arrascaeta'))
    .select('temporada', 'game_id', 'player', 'minutos_jogados')
    .orderBy('temporada', 'game_id')
)

In [0]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.regression import GBTRegressor
from pyspark.ml import Pipeline

print("1. Treinando Rede Neural de Risco de Gol (dVAEP & oVAEP)...")
df_ml_treino = df_ml_contexto.withColumn('is_gol', F.when((F.col('type_id') == 11) & (F.col('result_id') == 1), 1).otherwise(0)).fillna(0)
assembler = VectorAssembler(inputCols=['start_x', 'x_destino_anterior', 'delta_tempo_segundos', 'recuperou_posse'], outputCol="features")
gbt = GBTClassifier(labelCol="is_gol", featuresCol="features", maxIter=20)
modelo_treinado = Pipeline(stages=[assembler, gbt]).fit(df_ml_treino)

extrair_prob = F.udf(lambda v: float(v[1]), "double")
df_previsoes = modelo_treinado.transform(df_ml_treino)
df_probabilidades = df_previsoes.withColumn("probabilidade_gol", extrair_prob(F.col("probability"))) \
                                .withColumn("dvaep_lance", F.col("probabilidade_gol") * F.when(F.col("recuperou_posse") == 1, 1.5).otherwise(-0.5))

print("2. Treinando Regressor de Ameaça Impedida (DAxT)...")
janela_jogo_ml = Window.partitionBy('game_id', 'period_id').orderBy('time_seconds')
df_ovaep = df_probabilidades.withColumn("prob_gol_atual", F.col("probabilidade_gol")) \
                            .withColumn("prob_gol_anterior", F.lag("probabilidade_gol", 1).over(janela_jogo_ml)).fillna({"prob_gol_anterior": 0.0}) \
                            .withColumn("ovaep_lance", F.when(F.col('type_id').isin([0, 1, 11, 21]), (F.col("prob_gol_atual") - F.col("prob_gol_anterior"))).otherwise(0.0))

df_daxt_contexto = df_ovaep.withColumn('xT_anterior_1', F.lag('prob_gol_atual', 1).over(janela_jogo_ml)) \
                           .withColumn('xT_anterior_2', F.lag('prob_gol_atual', 2).over(janela_jogo_ml)) \
                           .withColumn('x_anterior_1', F.lag('start_x', 1).over(janela_jogo_ml)) \
                           .withColumn('x_anterior_2', F.lag('start_x', 2).over(janela_jogo_ml)).fillna(0.0)

is_acao_defensiva = F.col('type_id').isin([8, 17])
assembler_daxt = VectorAssembler(inputCols=['xT_anterior_1', 'xT_anterior_2', 'x_anterior_1', 'x_anterior_2'], outputCol="features_daxt")
gbt_regressor = GBTRegressor(featuresCol="features_daxt", labelCol="prob_gol_atual", predictionCol="prediction_daxt", lossType="absolute", maxIter=20)

df_treino_daxt = assembler_daxt.transform(df_daxt_contexto)
modelo_daxt = gbt_regressor.fit(df_treino_daxt.filter(~is_acao_defensiva))
df_daxt_final = modelo_daxt.transform(df_treino_daxt).withColumn("daxt_lance", F.when(is_acao_defensiva, F.col("prediction_daxt")).otherwise(0.0))

print("Auditoria Célula 2: Avaliação de Risco Lance a Lance (Arrascaeta)")
display(
    df_daxt_final.filter(F.col('player').contains('Arrascaeta'))
    .select('game_id', 'period_id', 'time_seconds', 'type_id', 'probabilidade_gol', 'ovaep_lance', 'daxt_lance')
    .orderBy('game_id', 'period_id', 'time_seconds')
)

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window

print("1. Calculando Multiplicadores PADJ e P90 no Nível da Partida...")
# Filtra quem jogou pelo menos 15 minutos para evitar distorções estatísticas extremas (ex: 1 desarme em 1 minuto virar 90 no P90)
df_base_jogo = df_game_log_temp.filter(F.col('minutos_jogados') >= 15)

df_padj_jogo = df_base_jogo.withColumn('padj_des', ((F.col('desarmes') / F.col('minutos_jogados')) * 90) * (50 / (100 - F.col('posse_bola')))) \
                           .withColumn('padj_int', ((F.col('interceptacoes') / F.col('minutos_jogados')) * 90) * (50 / (100 - F.col('posse_bola')))) \
                           .withColumn('p90_cor', (F.col('cortes') / F.col('minutos_jogados')) * 90) \
                           .withColumn('p90_fal', (F.col('faltas') / F.col('minutos_jogados')) * 90) \
                           .withColumn('p90_dri', (F.col('driblado') / F.col('minutos_jogados')) * 90).fillna(0)

print("2. Calculando o Score 0-100 (Z-Score por Partida)...")
janela_global = Window.partitionBy()
df_impacto_jogo = df_padj_jogo.withColumn('z_des', (F.col('padj_des') - F.mean('padj_des').over(janela_global)) / F.stddev('padj_des').over(janela_global)) \
                              .withColumn('z_int', (F.col('padj_int') - F.mean('padj_int').over(janela_global)) / F.stddev('padj_int').over(janela_global)) \
                              .withColumn('z_cor', (F.col('p90_cor') - F.mean('p90_cor').over(janela_global)) / F.stddev('p90_cor').over(janela_global)) \
                              .withColumn('z_fal', (F.col('p90_fal') - F.mean('p90_fal').over(janela_global)) / F.stddev('p90_fal').over(janela_global)) \
                              .withColumn('z_dri', (F.col('p90_dri') - F.mean('p90_dri').over(janela_global)) / F.stddev('p90_dri').over(janela_global)) \
                              .withColumn('impacto_bruto_jogo', F.col('z_des') + F.col('z_int') + F.col('z_cor') - F.col('z_fal') - F.col('z_dri'))

janela_ranking_jogo = Window.orderBy('impacto_bruto_jogo')

# Aqui a mágica acontece: O Score é atrelado ao game_id específico!
df_score_100 = df_impacto_jogo.withColumn('score_defesa_100', F.round(F.percent_rank().over(janela_ranking_jogo) * 100, 1)) \
                              .select('game_id', 'player', 'team_id', 'score_defesa_100')

print("Auditoria Célula 3: Score Defensivo por Partida (Arrascaeta)")
display(
    df_score_100.filter(F.col('player').contains('Arrascaeta'))
    .select('game_id', 'player', 'team_id', 'score_defesa_100')
)

In [0]:
print("1. Consolidando Métricas Analíticas por Jogo...")
# Agrupa os valores de ML a nível da partida
df_dvaep_jogo = df_daxt_final.filter(F.col('type_id').isin([8, 10, 9, 18])).groupBy('game_id', 'player', 'team_id').agg(F.sum('dvaep_lance').alias('dvaep_partida'))
df_ovaep_jogo = df_daxt_final.groupBy('game_id', 'player', 'team_id').agg(F.sum('ovaep_lance').alias('ovaep_partida'))
df_daxt_jogo = df_daxt_final.groupBy('game_id', 'player', 'team_id').agg(F.sum('daxt_lance').alias('daxt_partida'))

# GK Score (Baseado nos volumes originais)
df_gk_metricas = df_daxt_final.filter(F.col('type_id').isin([0, 14, 15, 16, 17])).groupBy('game_id', 'player', 'team_id').agg(
    F.sum(F.when((F.col('type_id') == 0) & (F.col('result_id') == 1), 1).otherwise(0)).alias('gk_passes_certos'),
    F.sum(F.when((F.col('type_id') == 0) & (F.col('result_id') == 0), 1).otherwise(0)).alias('gk_passes_errados'),
    F.sum(F.when(F.col('type_id').isin([14, 15, 16, 17]), 1).otherwise(0)).alias('gk_volume_defesas')
)

print("2. Juntando as Peças e Escalonando as Probabilidades (x100)...")
df_dashboard = df_game_log_temp.join(df_dvaep_jogo, on=['game_id', 'player', 'team_id'], how='left') \
                               .join(df_ovaep_jogo, on=['game_id', 'player', 'team_id'], how='left') \
                               .join(df_daxt_jogo, on=['game_id', 'player', 'team_id'], how='left') \
                               .join(df_gk_metricas, on=['game_id', 'player', 'team_id'], how='left') \
                               .join(df_score_100, on=['game_id', 'player', 'team_id'], how='left') \
                               .fillna({
                                   'dvaep_partida': 0, 'ovaep_partida': 0, 'daxt_partida': 0,
                                   'gk_passes_certos': 0, 'gk_passes_errados': 0, 'gk_volume_defesas': 0,
                                   'score_defesa_100': 0
                               })


# Multiplica a Inteligência Artificial por 100 para virar "Pontos" visualmente agradáveis no Dashboard
df_dashboard = df_dashboard.withColumn('dvaep_partida', F.round(F.col('dvaep_partida') * 100, 2)) \
                           .withColumn('ovaep_partida', F.round(F.col('ovaep_partida') * 100, 2)) \
                           .withColumn('daxt_partida', F.round(F.col('daxt_partida') * 100, 2))

print("Auditoria Célula 4: Tabela Matriz Consolidada Pronta (Arrascaeta)")
display(
    df_dashboard.filter(F.col('player').contains('Arrascaeta'))
    .select('temporada', 'game_id', 'player', 'minutos_jogados', 'dvaep_partida', 'ovaep_partida', 'score_defesa_100')
    .orderBy('temporada', 'game_id')
)

print("3. Exportando o Banco Consolidado...")
caminho_exportacao = "/Volumes/workspace/default/parquet/dashboard_final.parquet"
df_final_pandas = df_dashboard.toPandas()
df_final_pandas.to_parquet(caminho_exportacao, index=False)
print(f"✅ EXPORTAÇÃO CONCLUÍDA! Pronto para o download.")